# Email Sending

## 1 - Raw SMTP (standard library)

### Using Google SMTP

- For this setup, we need to generate the `Google App Password`.
- The Gmail Account must be 2-FA (Two Factor Authication) protected to generate the Google App Password.
- Rename the `.env.sample` file to `.env` and enter the values for each environment variable.
- `SMTP_HOST` is google's SMTP host name which is `smtp.gmail.com`.
- `SMTP_PORT` is google's SMTP port which is `587`.
- `SMTP_USER` is the email address used to generate the `Google App Password`.
- `SMTP_PASS` is the password provided by the `Google App Password` manager.

In [ ]:
# ! pip install email-validator # uncomment this to install package via pip
# ! pip install python-dotenv # uncomment this to install package via pip 
! uv add email-validator
! uv add python-dotenv

In [ ]:
# imports
import os
import smtplib
import ssl
import time
from dotenv import load_dotenv
from email.message import EmailMessage
from typing import List, Optional

In [ ]:
# constants

SMTP_HOST: str = os.getenv("SMTP_HOST", "")
SMTP_PORT: int = int(os.getenv("SMTP_PORT", 0))
SMTP_USER: str = os.getenv("SMTP_USER", "")
SMTP_PASS: str = os.getenv("SMTP_PASS", "")

RETRY_ATTEMPTS: int = 3
RETRY_BACKOFF: int = 2  # in seconds

In [ ]:
# load environment variables
load_dotenv(".env")

In [ ]:
class SMTPEmailSender:
    def __init__(self, host: str, port: int, user: str, password: str):
        self.host = host
        self.port = port
        self.user = user
        self.password = password

    def _build_message(
        self, subject: str, body: str, to: List[str], cc: Optional[List[str]] = None
    ) -> EmailMessage:
        email_message: EmailMessage = EmailMessage()
        email_message["From"] = self.user
        email_message["To"] = ", ".join(to)
        if cc:
            email_message["Cc"] = ", ".join(cc)
        email_message["Subject"] = subject
        email_message.set_content(body)
        return email_message

    def send_email(
        self, subject: str, body: str, to: List[str], cc: Optional[List[str]] = None
    ) -> None:
        email_message: EmailMessage = self._build_message(
            subject=subject, body=body, to=to, cc=cc
        )
        recipients: List[str] = to + (cc or [])

        # create secure channel with the smtp
        context: ssl.SSLContext = ssl.create_default_context()

        previous_exception = None

        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                with smtplib.SMTP(host=self.host, port=self.port, timeout=10) as server:
                    server.ehlo()  # this is optional if we don't add this, the new features of the smtp will not be available
                    server.starttls(context=context)
                    server.ehlo()
                    server.login(self.user, self.password)
                    server.send_message(
                        msg=email_message, from_addr=self.user, to_addrs=recipients
                    )
                return  # success
            except Exception as e:
                previous_exception = e
                time.sleep(RETRY_BACKOFF * attempt)
        raise RuntimeError(f"Email sending failed after retries: {previous_exception}")

In [ ]:
if __name__ == "__main__":
    sender = SMTPEmailSender(
        host=SMTP_HOST, port=SMTP_PORT, user=SMTP_USER, password=SMTP_PASS
    )
    sender.send_email(
        subject="Test Email",
        body="Hello from production-grade SMTP sender",
        to=["pervaix.fp@gmail.com"],
    )